# 02 — Customer EDA

**Project:** Food Delivery Operations Analytics  
**Author:** Sitanshu Singh

Exploring customer behavior — ordering frequency, average spend, weekday vs weekend patterns, and cancellations.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 5)

orders = pd.read_csv('../data/cleaned/orders_cleaned.csv', parse_dates=['order_date'])
customers = pd.read_csv('../data/cleaned/customers_cleaned.csv')

print(f'Loaded {len(orders)} orders, {len(customers)} customers')

## 1. Order Frequency Distribution

In [ ]:
order_counts = orders.groupby('customer_id').size().reset_index(name='order_count')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(order_counts['order_count'], bins=40, color='#f97316', edgecolor='white')
axes[0].set_title('Order Count Distribution per Customer')
axes[0].set_xlabel('Number of Orders')
axes[0].set_ylabel('Number of Customers')

# Pareto curve
sorted_counts = order_counts.sort_values('order_count', ascending=False)
sorted_counts['cumulative_orders'] = sorted_counts['order_count'].cumsum()
sorted_counts['cumulative_pct'] = sorted_counts['cumulative_orders'] / sorted_counts['order_count'].sum() * 100
sorted_counts['customer_pct'] = range(1, len(sorted_counts)+1)
sorted_counts['customer_pct'] = sorted_counts['customer_pct'] / len(sorted_counts) * 100

axes[1].plot(sorted_counts['customer_pct'], sorted_counts['cumulative_pct'], color='#f97316')
axes[1].axvline(x=20, color='gray', linestyle='--', alpha=0.7)
axes[1].axhline(y=61, color='gray', linestyle='--', alpha=0.7)
axes[1].set_title('Pareto Curve — top 20% of customers = 61% of orders')
axes[1].set_xlabel('% of Customers')
axes[1].set_ylabel('% of Total Orders')

plt.tight_layout()
plt.savefig('../reports/customer_pareto.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Weekend vs Weekday Orders

In [ ]:
orders['day_type'] = orders['is_weekend'].map({True: 'Weekend', False: 'Weekday'})
day_type_counts = orders.groupby('day_type').size()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(day_type_counts.index, day_type_counts.values, color=['#f97316', '#1e3a5f'], width=0.5)
ax.set_title('Weekday vs Weekend Order Volume')
ax.set_ylabel('Number of Orders')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{bar.get_height():,}', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('../reports/weekday_vs_weekend.png', dpi=150, bbox_inches='tight')
plt.show()

spike = (day_type_counts['Weekend'] - day_type_counts['Weekday']) / day_type_counts['Weekday'] * 100
print(f'Weekend orders are {spike:.1f}% higher than weekday orders')

## 3. Order Value Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
delivered = orders[orders['status'] == 'Delivered']
ax.hist(delivered['order_value'], bins=60, color='#f97316', edgecolor='white', alpha=0.8)
ax.axvline(delivered['order_value'].mean(), color='#1e3a5f', linestyle='--', label=f'Mean: ₹{delivered["order_value"].mean():.0f}')
ax.axvline(delivered['order_value'].median(), color='red', linestyle='--', label=f'Median: ₹{delivered["order_value"].median():.0f}')
ax.set_title('Order Value Distribution')
ax.set_xlabel('Order Value (₹)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()